In [10]:
import os

import joblib
import numpy as np
import pandas as pd

# CatBoost <1.2.8 lacks __sklearn_tags__ required by scikit-learn >=1.6
import sklearn
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from packaging.version import Version
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

if Version(sklearn.__version__) >= Version("1.6") and not hasattr(CatBoostClassifier, "__sklearn_tags__"):
    from sklearn.utils._tags import ClassifierTags, InputTags, Tags, TargetTags
    def _catboost_sklearn_tags(self):
        return Tags(
            estimator_type="classifier",
            target_tags=TargetTags(required=True),
            classifier_tags=ClassifierTags(),
            input_tags=InputTags(),
        )
    CatBoostClassifier.__sklearn_tags__ = _catboost_sklearn_tags

RANDOM_STATE = 42
CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)


In [11]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print(f"Training data: {X_train.shape}")
print(f"Test data: {X_test.shape}")

Training data: (4930, 45)
Test data: (1057, 45)


In [12]:
models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),

    "decision_tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "xgboost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "lightgbm": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    ),

    "catboost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        verbose=False,
        random_seed=RANDOM_STATE
    )
}

In [13]:
scoring = {
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

cv_results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=CV,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "model": name,
        "roc_auc": scores["test_roc_auc"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean()
    })

cv_results = (
    pd.DataFrame(cv_results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

cv_results

,model,roc_auc,precision,recall,f1
0,logistic_regression,0.844338,0.654588,0.548135,0.595995
1,catboost,0.843554,0.665645,0.525200,0.586214
2,lightgbm,0.830186,0.627205,0.520643,0.568661
3,random_forest,0.826367,0.639345,0.498450,0.559993
4,decision_tree,0.668024,0.500365,0.522945,0.511303
5,xgboost,NaN,0.662281,0.527490,0.586637


In [14]:
top_models = cv_results.head(3)["model"].tolist()

param_grids = {
    "xgboost": {
        "n_estimators": [200, 300, 500],
        "max_depth": [3, 4, 5, 6],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.7, 0.8, 1.0]
    },

    "lightgbm": {
        "n_estimators": [200, 300, 500],
        "learning_rate": [0.01, 0.05, 0.1],
        "num_leaves": [15, 31, 63],
        "max_depth": [-1, 5, 10],
        "subsample": [0.7, 0.8, 1.0]
    },

    "catboost": {
        "iterations": [200, 300, 500],
        "depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5, 7]
    }
}

tuned_models = {}
tuning_results = []

for name in top_models:

    if name not in param_grids:
        continue

    search = RandomizedSearchCV(
        estimator=models[name],
        param_distributions=param_grids[name],
        n_iter=15,
        scoring="roc_auc",
        cv=CV,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True
    )

    search.fit(X_train, y_train)

    tuned_models[name] = search.best_estimator_

    tuning_results.append({
        "model": name,
        "best_cv_roc_auc": search.best_score_,
        "best_params": search.best_params_
    })

tuning_results = (
    pd.DataFrame(tuning_results)
    .sort_values("best_cv_roc_auc", ascending=False)
    .reset_index(drop=True)
)

tuning_results

,model,best_cv_roc_auc,best_params
0,catboost,0.848814,"{'learning_rate': 0.05, 'l2_leaf_reg': 7, 'ite..."
1,lightgbm,0.845816,"{'subsample': 0.7, 'num_leaves': 15, 'n_estima..."


In [15]:
final_results = []

for name, model in tuned_models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=CV,
        scoring=scoring,
        n_jobs=-1
    )

    final_results.append({
        "model": name,
        "roc_auc": scores["test_roc_auc"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean()
    })

final_results = (
    pd.DataFrame(final_results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

final_results

,model,roc_auc,precision,recall,f1
0,catboost,0.848814,0.674575,0.537422,0.597509
1,lightgbm,0.845816,0.662446,0.532111,0.590038


In [16]:
best_model_name = final_results.loc[0, "model"]
final_model = tuned_models[best_model_name]

print(f"Selected model: {best_model_name}")
print(f"CV ROC-AUC: {final_results.loc[0, 'roc_auc']:.4f}")

Selected model: catboost
CV ROC-AUC: 0.8488


In [17]:
model_path = os.path.join(
    MODEL_DIR,
    "telco_churn_model.joblib"
)

joblib.dump(final_model, model_path)

print(f"Model saved to: {model_path}")

Model saved to: ../models\telco_churn_model.joblib


In [18]:
metadata = {
    "model": best_model_name,
    "cv_strategy": "StratifiedKFold",
    "cv_folds": 5,
    "primary_metric": "roc_auc",
    "random_state": RANDOM_STATE
}

joblib.dump(
    metadata,
    os.path.join(MODEL_DIR, "model_metadata.joblib")
)

['../models\\model_metadata.joblib']